# Parallel Asteroid Belt Dynamics & Planet Formation Simulation

Serial N-body simulation modeling gravitational interactions, orbital mechanics, collisions, and planet formation in a protoplanetary disk.

In [1]:
%%writefile nbody_serial.cpp
// =============================================================================
// Parallel Asteroid Belt Dynamics & Planet Formation Simulation
// Serial C++ Implementation — Velocity-Verlet N-body with Collisions
// =============================================================================

#include <iostream>
#include <fstream>
#include <vector>
#include <cmath>
#include <cstdlib>
#include <string>
#include <algorithm>
#include <chrono>
#include <sstream>
#include <iomanip>

// ========================== CONSTANTS ========================================
const double G             = 6.674e-11;   // gravitational constant (SI)
const double PI            = 3.14159265358979323846;
const double DENSITY       = 3000.0;      // body density kg/m^3 (rocky)

// Collision cross-section inflation factor.
// Physical radii (~35 km) are negligible at AU scales, so collisions
// essentially never occur in feasible simulation times. Real planet
// formation codes (REBOUND, MERCURY) inflate radii to accelerate
// accretion. This multiplier is applied to asteroid radii ONLY.
const double RADIUS_INFLATION = 100.0;

// ========================== DATA STRUCTURES ==================================
struct Body {
    double x, y;       // position [m]
    double vx, vy;     // velocity [m/s]
    double ax, ay;     // acceleration [m/s^2]
    double mass;       // [kg]
    double radius;     // [m]
    bool   alive;      // false if merged into another body
};

// ========================== SIMULATION PARAMETERS ============================
struct SimParams {
    int    num_bodies;
    double star_mass;
    double disk_inner;       // inner edge of disk [m]
    double disk_outer;       // outer edge of disk [m]
    double dt;               // timestep [s]
    int    num_steps;
    int    snapshot_interval; // save every N steps
    double softening;        // softening length [m]
    double velocity_perturbation; // fractional perturbation on Keplerian velocity
    std::string output_dir;
};

// ========================== UTILITY ==========================================
double compute_radius(double mass) {
    // Sphere: m = (4/3) * pi * rho * r^3  =>  r = (3m / 4*pi*rho)^(1/3)
    return std::pow(3.0 * mass / (4.0 * PI * DENSITY), 1.0 / 3.0);
}

// ========================== INITIAL CONDITIONS ===============================
// Generate a Keplerian protoplanetary disk
std::vector<Body> generate_initial_conditions(const SimParams& params) {
    std::vector<Body> bodies;
    bodies.reserve(params.num_bodies + 1);

    // Central star (index 0)
    Body star;
    star.x  = 0.0; star.y  = 0.0;
    star.vx = 0.0; star.vy = 0.0;
    star.ax = 0.0; star.ay = 0.0;
    star.mass   = params.star_mass;
    star.radius = compute_radius(params.star_mass);
    star.alive  = true;
    bodies.push_back(star);

    // Disk bodies
    std::srand(42); // reproducible
    double total_disk_mass = 0.01 * params.star_mass; // disk is 1% of star mass
    double body_mass = total_disk_mass / params.num_bodies;

    for (int i = 0; i < params.num_bodies; ++i) {
        Body b;
        // Uniform distribution in radius (biased toward outer disk for area coverage)
        double u = (double)std::rand() / RAND_MAX;
        double r = params.disk_inner + u * (params.disk_outer - params.disk_inner);

        // Random angle
        double theta = 2.0 * PI * ((double)std::rand() / RAND_MAX);

        b.x = r * std::cos(theta);
        b.y = r * std::sin(theta);

        // Keplerian circular velocity
        double v_kep = std::sqrt(G * params.star_mass / r);

        // Tangential direction (perpendicular to radial, counter-clockwise)
        double vx_tan = -v_kep * std::sin(theta);
        double vy_tan =  v_kep * std::cos(theta);

        // Add perturbation
        double perturb_vx = params.velocity_perturbation * v_kep *
                            (2.0 * ((double)std::rand() / RAND_MAX) - 1.0);
        double perturb_vy = params.velocity_perturbation * v_kep *
                            (2.0 * ((double)std::rand() / RAND_MAX) - 1.0);

        b.vx = vx_tan + perturb_vx;
        b.vy = vy_tan + perturb_vy;

        b.ax = 0.0; b.ay = 0.0;

        // Small mass variation (0.5x to 2x)
        double mass_factor = 0.5 + 1.5 * ((double)std::rand() / RAND_MAX);
        b.mass   = body_mass * mass_factor;
        b.radius = compute_radius(b.mass) * RADIUS_INFLATION;
        b.alive  = true;

        bodies.push_back(b);
    }

    return bodies;
}

// ========================== PHYSICS: FORCE COMPUTATION =======================
// Compute gravitational accelerations — O(N^2) all-pairs
void compute_forces(std::vector<Body>& bodies, double softening_sq) {
    int n = bodies.size();

    // Reset accelerations
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        bodies[i].ax = 0.0;
        bodies[i].ay = 0.0;
    }

    // All-pairs force computation
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        for (int j = i + 1; j < n; ++j) {
            if (!bodies[j].alive) continue;

            double dx = bodies[j].x - bodies[i].x;
            double dy = bodies[j].y - bodies[i].y;
            double dist_sq = dx * dx + dy * dy + softening_sq;
            double inv_dist = 1.0 / std::sqrt(dist_sq);
            double inv_dist3 = inv_dist * inv_dist * inv_dist;

            // Force magnitude / distance = G * m_j / r^3 (for acceleration on i)
            double fx = G * dx * inv_dist3;
            double fy = G * dy * inv_dist3;

            // Newton's 3rd law: apply equal-opposite
            bodies[i].ax += fx * bodies[j].mass;
            bodies[i].ay += fy * bodies[j].mass;
            bodies[j].ax -= fx * bodies[i].mass;
            bodies[j].ay -= fy * bodies[i].mass;
        }
    }
}

// ========================== INTEGRATOR: VELOCITY-VERLET ======================
void integrate_step(std::vector<Body>& bodies, double dt, double softening_sq) {
    int n = bodies.size();

    // Step 1: Half-kick (update velocities by half-step using current accelerations)
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        bodies[i].vx += 0.5 * dt * bodies[i].ax;
        bodies[i].vy += 0.5 * dt * bodies[i].ay;
    }

    // Step 2: Drift (update positions using half-kicked velocities)
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        bodies[i].x += dt * bodies[i].vx;
        bodies[i].y += dt * bodies[i].vy;
    }

    // Step 3: Compute new forces at the new positions
    compute_forces(bodies, softening_sq);

    // Step 4: Half-kick again (complete velocity update)
    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        bodies[i].vx += 0.5 * dt * bodies[i].ax;
        bodies[i].vy += 0.5 * dt * bodies[i].ay;
    }
}

// ========================== COLLISION DETECTION & MERGING ====================
int handle_collisions(std::vector<Body>& bodies) {
    int n = bodies.size();
    int merges = 0;

    for (int i = 0; i < n; ++i) {
        if (!bodies[i].alive) continue;
        if (i == 0) continue;  // star does not participate in merging
        for (int j = i + 1; j < n; ++j) {
            if (!bodies[j].alive) continue;

            double dx = bodies[j].x - bodies[i].x;
            double dy = bodies[j].y - bodies[i].y;
            double dist = std::sqrt(dx * dx + dy * dy);
            double overlap_dist = bodies[i].radius + bodies[j].radius;

            if (dist < overlap_dist) {
                // Merge j into i (keep the more massive body's index)
                double m1 = bodies[i].mass;
                double m2 = bodies[j].mass;
                double m_total = m1 + m2;

                // Momentum-conserving velocity
                bodies[i].vx = (m1 * bodies[i].vx + m2 * bodies[j].vx) / m_total;
                bodies[i].vy = (m1 * bodies[i].vy + m2 * bodies[j].vy) / m_total;

                // Center of mass position (weighted)
                bodies[i].x = (m1 * bodies[i].x + m2 * bodies[j].x) / m_total;
                bodies[i].y = (m1 * bodies[i].y + m2 * bodies[j].y) / m_total;

                bodies[i].mass   = m_total;
                bodies[i].radius = compute_radius(m_total) * RADIUS_INFLATION;

                // Kill body j
                bodies[j].alive = false;
                ++merges;
            }
        }
    }
    return merges;
}

// ========================== I/O: CSV OUTPUT ==================================
void write_snapshot(const std::vector<Body>& bodies, int step, const std::string& dir) {
    std::ostringstream filename;
    filename << dir << "/step_" << std::setw(6) << std::setfill('0') << step << ".csv";

    std::ofstream file(filename.str());
    if (!file.is_open()) {
        std::cerr << "ERROR: Cannot open " << filename.str() << std::endl;
        return;
    }

    file << "id,x,y,vx,vy,mass,radius\n";
    int id = 0;
    for (size_t i = 0; i < bodies.size(); ++i) {
        if (!bodies[i].alive) continue;
        file << id << ","
             << std::scientific << std::setprecision(8)
             << bodies[i].x << ","
             << bodies[i].y << ","
             << bodies[i].vx << ","
             << bodies[i].vy << ","
             << bodies[i].mass << ","
             << bodies[i].radius << "\n";
        ++id;
    }
    file.close();
}

// Count alive bodies
int count_alive(const std::vector<Body>& bodies) {
    int count = 0;
    for (size_t i = 0; i < bodies.size(); ++i) {
        if (bodies[i].alive) ++count;
    }
    return count;
}

// ========================== MAIN =============================================
int main(int argc, char* argv[]) {
    // ----- Default simulation parameters -----
    SimParams params;
    params.num_bodies             = 2000;
    params.star_mass              = 1.989e30;        // 1 solar mass
    params.disk_inner             = 1.0e11;           // ~0.67 AU
    params.disk_outer             = 5.0e11;           // ~3.3 AU
    params.dt                     = 1.0e6;            // ~11.6 days
    params.num_steps              = 200000;
    params.snapshot_interval      = 10;
    params.softening              = 1.0e9;            // softening length
    params.velocity_perturbation  = 0.10;
    params.output_dir             = "data_serial";

    // Simple command-line overrides
    for (int i = 1; i < argc; ++i) {
        std::string arg = argv[i];
        if (arg == "--bodies"    && i+1 < argc) params.num_bodies        = std::atoi(argv[++i]);
        else if (arg == "--steps"     && i+1 < argc) params.num_steps         = std::atoi(argv[++i]);
        else if (arg == "--dt"        && i+1 < argc) params.dt               = std::atof(argv[++i]);
        else if (arg == "--interval"  && i+1 < argc) params.snapshot_interval = std::atoi(argv[++i]);
        else if (arg == "--outdir"    && i+1 < argc) params.output_dir       = argv[++i];
    }

    double softening_sq = params.softening * params.softening;

    std::cout << "=== N-Body Asteroid Belt Simulation (Serial) ===" << std::endl;
    std::cout << "Bodies: " << params.num_bodies << std::endl;
    std::cout << "Steps:  " << params.num_steps  << std::endl;
    std::cout << "dt:     " << params.dt << " s" << std::endl;
    std::cout << "Output: " << params.output_dir << "/" << std::endl;

    // Create output directory
    std::string cmd = "mkdir -p " + params.output_dir;
    system(cmd.c_str());

    // Generate initial conditions
    auto bodies = generate_initial_conditions(params);
    std::cout << "Generated " << bodies.size() << " bodies (1 star + "
              << params.num_bodies << " asteroids)" << std::endl;

    // Initial force computation (needed for Verlet first half-kick)
    compute_forces(bodies, softening_sq);

    // Write initial snapshot
    write_snapshot(bodies, 0, params.output_dir);

    // ----- Main simulation loop -----
    auto t_start = std::chrono::high_resolution_clock::now();

    for (int step = 1; step <= params.num_steps; ++step) {
        // Velocity-Verlet integration step
        integrate_step(bodies, params.dt, softening_sq);

        // Collision detection and merging
        int merges = handle_collisions(bodies);

        // Snapshot output
        if (step % params.snapshot_interval == 0) {
            write_snapshot(bodies, step, params.output_dir);
        }

        // Progress reporting
        if (step % 100 == 0 || step == params.num_steps) {
            int alive = count_alive(bodies);
            std::cout << "Step " << step << "/" << params.num_steps
                      << " | Bodies alive: " << alive;
            if (merges > 0) std::cout << " | Merges this step: " << merges;
            std::cout << std::endl;
        }
    }

    auto t_end = std::chrono::high_resolution_clock::now();
    double elapsed = std::chrono::duration<double>(t_end - t_start).count();

    std::cout << "\n=== Simulation Complete ===" << std::endl;
    std::cout << "Total time: " << std::fixed << std::setprecision(3) << elapsed << " s" << std::endl;
    std::cout << "Final alive bodies: " << count_alive(bodies) << std::endl;
    std::cout << "Snapshots saved to: " << params.output_dir << "/" << std::endl;

    return 0;
}

Writing nbody_serial.cpp


### Compile and Run

In [2]:
!g++ -O3 -std=c++17 -o nbody_serial nbody_serial.cpp -lm
print("Compilation successful!")

nbody_serial.cpp: In function ‘int main(int, char**)’:
nbody_serial.cpp:302:11: warning: ignoring return value of ‘int system(const char*)’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
  302 |     system(cmd.c_str());
      |     ~~~~~~^~~~~~~~~~~~~
Compilation successful!


In [3]:
!./nbody_serial --bodies 2000 --steps 200000 --interval 10 --outdir data_serial

=== N-Body Asteroid Belt Simulation (Serial) ===
Bodies: 2000
Steps:  200000
dt:     1e+06 s
Output: data_serial/
Generated 2001 bodies (1 star + 2000 asteroids)
Step 100/200000 | Bodies alive: 791 | Merges this step: 1
Step 200/200000 | Bodies alive: 459 | Merges this step: 2
Step 300/200000 | Bodies alive: 321 | Merges this step: 1
Step 400/200000 | Bodies alive: 248 | Merges this step: 1
Step 500/200000 | Bodies alive: 204
Step 600/200000 | Bodies alive: 159
Step 700/200000 | Bodies alive: 131 | Merges this step: 1
Step 800/200000 | Bodies alive: 111
Step 900/200000 | Bodies alive: 95
Step 1000/200000 | Bodies alive: 83
Step 1100/200000 | Bodies alive: 75
Step 1200/200000 | Bodies alive: 68
Step 1300/200000 | Bodies alive: 65
Step 1400/200000 | Bodies alive: 60
Step 1500/200000 | Bodies alive: 56
Step 1600/200000 | Bodies alive: 51
Step 1700/200000 | Bodies alive: 46 | Merges this step: 1
Step 1800/200000 | Bodies alive: 43
Step 1900/200000 | Bodies alive: 41
Step 2000/200000 | Bodi

## Visualization

In [4]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Circle
from matplotlib.collections import PathCollection
import os
import glob
import csv
from IPython.display import HTML, Video
import warnings
warnings.filterwarnings('ignore')

# ========================== CONFIGURATION ====================================
DATA_DIR = "data_serial"
OUTPUT_VIDEO = "planet_formation_serial.mp4"
FPS = 30
DPI = 150
FIGSIZE = (10, 10)

# Physical scale for plot limits (in AU for display, data is in meters)
AU = 1.496e11  # 1 AU in meters
PLOT_LIMIT_AU = 4.0  # show ±4 AU

# ========================== LOAD SNAPSHOTS ====================================
def load_snapshot(filepath):
    """Load a single CSV snapshot, return arrays of x, y, mass, radius."""
    x, y, mass, radius = [], [], [], []
    with open(filepath, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            x.append(float(row['x']))
            y.append(float(row['y']))
            mass.append(float(row['mass']))
            radius.append(float(row['radius']))
    return np.array(x), np.array(y), np.array(mass), np.array(radius)

def get_sorted_snapshots(data_dir):
    """Get list of snapshot files sorted by step number."""
    files = glob.glob(os.path.join(data_dir, "step_*.csv"))
    files.sort(key=lambda f: int(os.path.basename(f).split('_')[1].split('.')[0]))
    return files

# Load all snapshots
snapshot_files = get_sorted_snapshots(DATA_DIR)
print(f"Found {len(snapshot_files)} snapshots in {DATA_DIR}/")

# Quick peek at first and last snapshot
x0, y0, m0, r0 = load_snapshot(snapshot_files[0])
xf, yf, mf, rf = load_snapshot(snapshot_files[-1])
print(f"Initial: {len(x0)} bodies, mass range [{m0.min():.2e}, {m0.max():.2e}] kg")
print(f"Final:   {len(xf)} bodies, mass range [{mf.min():.2e}, {mf.max():.2e}] kg")

Found 20001 snapshots in data_serial/
Initial: 2001 bodies, mass range [4.98e+24, 1.99e+30] kg
Final:   7 bodies, mass range [1.49e+25, 1.99e+30] kg


In [ ]:
import matplotlib.animation as animation
import sys

# ========================== COLOR MAP =========================================
# Mass-based coloring: small=blue/gray → medium=orange → large=white/yellow
def mass_to_color(masses, global_min_mass, global_max_mass):
    """Map mass values to RGBA colors for astrophysics-style rendering."""
    if global_max_mass <= global_min_mass:
        norm = np.zeros_like(masses)
    else:
        # Log-scale normalization for mass
        log_m = np.log10(np.clip(masses, global_min_mass, global_max_mass))
        log_min = np.log10(global_min_mass)
        log_max = np.log10(global_max_mass)
        if log_max > log_min:
            norm = (log_m - log_min) / (log_max - log_min)
        else:
            norm = np.zeros_like(log_m)

    # Custom colormap: dark blue → orange → bright yellow/white
    colors = np.zeros((len(masses), 4))
    for i, t in enumerate(norm):
        if t < 0.3:
            # Small bodies: steel blue / gray
            s = t / 0.3
            colors[i] = [0.4 + 0.2*s, 0.5 + 0.1*s, 0.7 - 0.1*s, 0.6 + 0.2*s]
        elif t < 0.7:
            # Medium bodies: orange
            s = (t - 0.3) / 0.4
            colors[i] = [0.8 + 0.2*s, 0.5 + 0.2*s, 0.2 - 0.1*s, 0.85 + 0.1*s]
        else:
            # Large bodies: bright yellow → white (planet embryos)
            s = (t - 0.7) / 0.3
            colors[i] = [1.0, 0.8 + 0.2*s, 0.3 + 0.7*s, 1.0]
    return colors

def mass_to_size(masses, global_min_mass, global_max_mass):
    """Map mass to marker size (proportional to radius ~ m^(1/3))."""
    if global_max_mass <= global_min_mass:
        return np.full_like(masses, 2.0)
    log_m = np.log10(np.clip(masses, global_min_mass, global_max_mass))
    log_min = np.log10(global_min_mass)
    log_max = np.log10(global_max_mass)
    if log_max > log_min:
        norm = (log_m - log_min) / (log_max - log_min)
    else:
        norm = np.zeros_like(log_m)
    # Size range: 1 to 120 points
    sizes = 1.0 + 150.0 * (norm ** 1.5)
    return sizes

# ========================== COMPUTE GLOBAL RANGES =============================
# We need global min/max mass across ALL snapshots for consistent coloring
print("Computing global mass range across all snapshots...")
all_masses = []
for sf in snapshot_files[::max(1, len(snapshot_files)//20)]:  # sample every ~20th
    _, _, m, _ = load_snapshot(sf)
    all_masses.append(m)
all_masses = np.concatenate(all_masses)
# Exclude the star (largest mass) for asteroid coloring
star_mass = all_masses.max()
asteroid_masses = all_masses[all_masses < star_mass * 0.5]
GLOBAL_MIN_MASS = asteroid_masses.min() if len(asteroid_masses) > 0 else all_masses.min()
GLOBAL_MAX_MASS = asteroid_masses.max() if len(asteroid_masses) > 0 else all_masses.max()
print(f"Asteroid mass range: [{GLOBAL_MIN_MASS:.2e}, {GLOBAL_MAX_MASS:.2e}] kg")
print(f"Star mass: {star_mass:.2e} kg")

# ========================== RENDER ANIMATION ==================================
total_frames = len(snapshot_files)
print(f"\nRendering animation ({total_frames} frames)...")

fig, ax = plt.subplots(1, 1, figsize=FIGSIZE, facecolor='black')
ax.set_facecolor('black')

# Store trail history
TRAIL_LENGTH = 5
trail_history = []

import time
_render_start_time = time.time()

def render_frame(frame_idx):
    ax.clear()
    ax.set_facecolor('black')

    # ---- Progress reporting ----
    frame_num = frame_idx + 1
    elapsed = time.time() - _render_start_time
    if frame_num > 1:
        fps_actual = (frame_num - 1) / elapsed
        eta = (total_frames - frame_num) / fps_actual
        eta_min, eta_sec = divmod(int(eta), 60)
        print(f"\rRendering frame {frame_num}/{total_frames} "
              f"({100*frame_num/total_frames:.1f}%) | "
              f"{fps_actual:.1f} frames/s | "
              f"ETA: {eta_min}m {eta_sec}s   ", end="")
    else:
        print(f"\rRendering frame {frame_num}/{total_frames} "
              f"({100*frame_num/total_frames:.1f}%)   ", end="")
    sys.stdout.flush()

    # Load snapshot
    x, y, mass, radius = load_snapshot(snapshot_files[frame_idx])

    # Convert to AU for display
    x_au = x / AU
    y_au = y / AU

    # Identify star (most massive body) vs asteroids
    star_idx = np.argmax(mass)

    # Separate star and asteroids
    is_star = np.zeros(len(mass), dtype=bool)
    is_star[star_idx] = True
    ast_mask = ~is_star

    # ---- Draw fading orbital trails ----
    trail_history.append((x_au[ast_mask].copy(), y_au[ast_mask].copy()))
    if len(trail_history) > TRAIL_LENGTH:
        trail_history.pop(0)

    for t_idx, (tx, ty) in enumerate(trail_history[:-1]):
        alpha = 0.05 + 0.1 * (t_idx / max(1, TRAIL_LENGTH))
        n_pts = min(len(tx), 300)  # limit trail points for performance
        ax.scatter(tx[:n_pts], ty[:n_pts], s=0.3, c='steelblue',
                   alpha=alpha, edgecolors='none', rasterized=True)

    # ---- Draw asteroids ----
    ast_x = x_au[ast_mask]
    ast_y = y_au[ast_mask]
    ast_mass = mass[ast_mask]

    colors = mass_to_color(ast_mass, GLOBAL_MIN_MASS, GLOBAL_MAX_MASS)
    sizes = mass_to_size(ast_mass, GLOBAL_MIN_MASS, GLOBAL_MAX_MASS)

    ax.scatter(ast_x, ast_y, s=sizes, c=colors, edgecolors='none',
               alpha=0.9, rasterized=True)

    # ---- Draw star with glow ----
    star_x = x_au[star_idx]
    star_y = y_au[star_idx]

    # Outer glow layers
    for glow_size, glow_alpha in [(800, 0.03), (500, 0.06), (300, 0.1), (150, 0.2)]:
        ax.scatter([star_x], [star_y], s=glow_size, c='gold',
                   alpha=glow_alpha, edgecolors='none', zorder=10)
    # Star core
    ax.scatter([star_x], [star_y], s=80, c='white', alpha=1.0,
               edgecolors='none', zorder=11)

    # ---- Axes and labels ----
    ax.set_xlim(-PLOT_LIMIT_AU, PLOT_LIMIT_AU)
    ax.set_ylim(-PLOT_LIMIT_AU, PLOT_LIMIT_AU)
    ax.set_aspect('equal')
    ax.set_xlabel('x [AU]', color='white', fontsize=12)
    ax.set_ylabel('y [AU]', color='white', fontsize=12)
    ax.tick_params(colors='gray', labelsize=9)
    for spine in ax.spines.values():
        spine.set_color('gray')
        spine.set_linewidth(0.5)

    # ---- Info overlay ----
    step_num = int(os.path.basename(snapshot_files[frame_idx]).split('_')[1].split('.')[0])
    n_bodies = int(ast_mask.sum())
    max_ast_mass = ast_mass.max() if len(ast_mass) > 0 else 0

    info_text = (f"Step: {step_num}\n"
                 f"Bodies: {n_bodies}\n"
                 f"Max asteroid mass: {max_ast_mass:.2e} kg")
    ax.text(0.02, 0.98, info_text, transform=ax.transAxes,
            color='white', fontsize=10, verticalalignment='top',
            fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.7))

    ax.set_title('Protoplanetary Disk — N-Body Simulation (Serial)',
                 color='white', fontsize=14, pad=10)

    return ax,

# Create animation
anim = animation.FuncAnimation(fig, render_frame,
                                frames=total_frames,
                                interval=1000//FPS, blit=False)

# Save as MP4
Writer = animation.FFMpegWriter(fps=FPS, bitrate=3000,
                                 extra_args=['-vcodec', 'libx264', '-pix_fmt', 'yuv420p'])
anim.save(OUTPUT_VIDEO, writer=Writer, dpi=DPI)
plt.close(fig)

total_elapsed = time.time() - _render_start_time
elapsed_min, elapsed_sec = divmod(int(total_elapsed), 60)
print(f"\n\nAnimation saved: {OUTPUT_VIDEO}")
print(f"Frames: {total_frames}, FPS: {FPS}, Duration: {total_frames/FPS:.1f}s")
print(f"Render time: {elapsed_min}m {elapsed_sec}s ({total_frames/total_elapsed:.1f} frames/s)")

Computing global mass range across all snapshots...
Asteroid mass range: [4.98e+24, 1.38e+28] kg
Star mass: 1.99e+30 kg

Rendering animation (20001 frames)...


In [ ]:
# Display the video in notebook (works on Colab)
from IPython.display import Video
Video(OUTPUT_VIDEO, embed=True, width=700)

### Static Comparison: Initial vs Final State

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8), facecolor='black')

for ax_idx, (snap_file, title) in enumerate([
    (snapshot_files[0], "Initial Disk (t=0)"),
    (snapshot_files[-1], f"Final State (step {len(snapshot_files)*5})")
]):
    ax = axes[ax_idx]
    ax.set_facecolor('black')

    x, y, mass, radius = load_snapshot(snap_file)
    x_au, y_au = x / AU, y / AU

    star_idx = np.argmax(mass)
    is_star = np.zeros(len(mass), dtype=bool)
    is_star[star_idx] = True
    ast_mask = ~is_star

    # Asteroids
    ast_mass = mass[ast_mask]
    colors = mass_to_color(ast_mass, GLOBAL_MIN_MASS, GLOBAL_MAX_MASS)
    sizes = mass_to_size(ast_mass, GLOBAL_MIN_MASS, GLOBAL_MAX_MASS)
    ax.scatter(x_au[ast_mask], y_au[ast_mask], s=sizes, c=colors,
               edgecolors='none', alpha=0.9, rasterized=True)

    # Star glow
    sx, sy = x_au[star_idx], y_au[star_idx]
    for gs, ga in [(800, 0.03), (500, 0.06), (300, 0.1), (150, 0.2)]:
        ax.scatter([sx], [sy], s=gs, c='gold', alpha=ga, edgecolors='none', zorder=10)
    ax.scatter([sx], [sy], s=80, c='white', alpha=1.0, edgecolors='none', zorder=11)

    ax.set_xlim(-PLOT_LIMIT_AU, PLOT_LIMIT_AU)
    ax.set_ylim(-PLOT_LIMIT_AU, PLOT_LIMIT_AU)
    ax.set_aspect('equal')
    ax.set_title(title, color='white', fontsize=14)
    ax.set_xlabel('x [AU]', color='white', fontsize=11)
    ax.set_ylabel('y [AU]', color='white', fontsize=11)
    ax.tick_params(colors='gray', labelsize=9)
    for spine in ax.spines.values():
        spine.set_color('gray')
        spine.set_linewidth(0.5)

    ax.text(0.02, 0.02, f"Bodies: {ast_mask.sum()}", transform=ax.transAxes,
            color='white', fontsize=10, fontfamily='monospace')

plt.tight_layout()
plt.savefig("serial_comparison.png", dpi=150, facecolor='black', bbox_inches='tight')
plt.show()
print("Saved: serial_comparison.png")

### Diagnostics: Body Count & Maximum Mass Evolution

In [ ]:
# Collect diagnostics from all snapshots
steps = []
body_counts = []
max_masses = []

for sf in snapshot_files:
    step_num = int(os.path.basename(sf).split('_')[1].split('.')[0])
    _, _, mass, _ = load_snapshot(sf)
    # Exclude star
    star_idx = np.argmax(mass)
    ast_mass = np.delete(mass, star_idx)

    steps.append(step_num)
    body_counts.append(len(ast_mass))
    max_masses.append(ast_mass.max() if len(ast_mass) > 0 else 0)

steps = np.array(steps)
body_counts = np.array(body_counts)
max_masses = np.array(max_masses)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Body count evolution
ax1.plot(steps, body_counts, 'c-', linewidth=1.5)
ax1.set_xlabel('Simulation Step', fontsize=12)
ax1.set_ylabel('Number of Bodies', fontsize=12)
ax1.set_title('Body Count Over Time (Mergers)', fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.fill_between(steps, body_counts, alpha=0.15, color='cyan')

# Maximum mass evolution
ax2.semilogy(steps, max_masses, 'orange', linewidth=1.5)
ax2.set_xlabel('Simulation Step', fontsize=12)
ax2.set_ylabel('Maximum Asteroid Mass [kg]', fontsize=12)
ax2.set_title('Largest Body Mass Over Time', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("serial_diagnostics.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"\nInitial bodies: {body_counts[0]} → Final bodies: {body_counts[-1]}")
print(f"Initial max mass: {max_masses[0]:.2e} kg → Final max mass: {max_masses[-1]:.2e} kg")
print(f"Mass growth factor: {max_masses[-1]/max_masses[0]:.1f}x")